# Pipeline de Geracao de Dataset Satelital (LANDSAT, MODIS, SENTINEL)

## Objetivo
Transformar chips `.tif` multibanda em arquivos `.csv` tabulares para treino de modelos de ML.

Cada linha do CSV representa uma imagem, com:
- `image_id`
- `filepath`
- pixels flatten por banda (`b{b}_p{i}`)

## Escopo deste notebook
1. leitura dos chips
2. padronizacao espacial para `256x256`
3. flatten por banda
4. escrita incremental em CSV
5. validacao de integridade dos arquivos gerados


## 1) Ambiente e Dependencias

Instala bibliotecas necessarias para leitura geoespacial e acompanhamento do processamento.


In [ ]:
!pip -q install rasterio tqdm

## 2) Conexao com Google Drive

Os dados de entrada e saida ficam no Drive. O mount e necessario para ler os chips e salvar os CSVs finais.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3) Configuracao de Paths

Define diretorios de entrada por sensor e pasta de saida dos datasets tabulares.

Verifique os `exists?` antes de iniciar o processamento.


In [7]:
from pathlib import Path

# ✅ AJUSTE para onde estão suas imagens no Drive
LANDSAT_DIR  = Path("/content/drive/MyDrive/SpectraAI/data_raw/LANDSAT_chips")
MODIS_DIR    = Path("/content/drive/MyDrive/SpectraAI/data_raw/MODIS_chips")
SENTINEL_DIR = Path("/content/drive/MyDrive/SpectraAI/data_raw/S2_chips")

# ✅ Pasta de saída (onde os CSVs serão salvos)
OUT_DIR = Path("/content/drive/MyDrive/SpectraAI/datasets_csv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("LANDSAT_DIR:", LANDSAT_DIR, "exists?", LANDSAT_DIR.exists())
print("MODIS_DIR:", MODIS_DIR, "exists?", MODIS_DIR.exists())
print("SENTINEL_DIR:", SENTINEL_DIR, "exists?", SENTINEL_DIR.exists())
print("OUT_DIR:", OUT_DIR)

LANDSAT_DIR: /content/drive/MyDrive/SpectraAI/data_raw/LANDSAT_chips exists? True
MODIS_DIR: /content/drive/MyDrive/SpectraAI/data_raw/MODIS_chips exists? True
SENTINEL_DIR: /content/drive/MyDrive/SpectraAI/data_raw/S2_chips exists? True
OUT_DIR: /content/drive/MyDrive/SpectraAI/datasets_csv


## 4) Funcoes Base do Pipeline

Funcoes implementadas nesta etapa:
- `center_crop_or_pad`: garante shape espacial fixo (`256x256`)
- `read_tif_as_chw`: leitura raster em formato `(C, H, W)`
- `build_header`: schema de colunas do CSV
- `folder_to_csv`: processamento de uma pasta completa para CSV

### Decisoes tecnicas
1. Patch fixo `256x256` para compatibilidade entre sensores.
2. Crop/pad central para preservar regiao central da cena.
3. Escrita incremental para reduzir risco de perda em execucoes longas.
4. `float32` para balancear precisao e uso de memoria.


In [ ]:
import numpy as np
import rasterio
from tqdm import tqdm
import csv
import os
import gc

PATCH = 256  # tamanho final

def center_crop_or_pad(img_chw, patch=256):
    """
    img_chw: np.array (C, H, W)
    - Se H/W maiores: recorta central
    - Se menores: faz pad com zeros no centro
    Retorna (C, patch, patch)
    """
    C, H, W = img_chw.shape

    # Pad se necessário
    pad_h = max(0, patch - H)
    pad_w = max(0, patch - W)
    if pad_h > 0 or pad_w > 0:
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        img_chw = np.pad(img_chw, ((0,0),(top,bottom),(left,right)), mode='constant', constant_values=0)
        C, H, W = img_chw.shape

    # Crop central se necessário
    start_h = (H - patch) // 2
    start_w = (W - patch) // 2
    return img_chw[:, start_h:start_h+patch, start_w:start_w+patch]

def read_tif_as_chw(path):
    """
    Lê um .tif com rasterio.
    Retorna np.array (C,H,W) em float32.
    """
    with rasterio.open(path) as src:
        arr = src.read()  # (bands, H, W)
    arr = arr.astype(np.float32)
    return arr

def build_header(num_bands, patch=256):
    """
    Cria nomes de colunas:
    b0_p0 ... b0_p65535, b1_p0 ... etc
    """
    n_pix = patch * patch
    header = ["image_id", "filepath"]
    for b in range(num_bands):
        header.extend([f"b{b}_p{i}" for i in range(n_pix)])
    return header

def folder_to_csv(
    folder: Path,
    out_csv: Path,
    patch=256,
    pattern="*.tif",
    limit=None,
    log_every=10
):
    """
    Percorre .tif em 'folder', transforma cada imagem numa linha e salva incrementalmente em CSV.
    """
    tif_files = sorted(folder.rglob(pattern))
    if limit is not None:
        tif_files = tif_files[:limit]

    if len(tif_files) == 0:
        raise ValueError(f"Nenhum arquivo .tif encontrado em: {folder}")

    print(f"\n📂 Pasta: {folder}")
    print(f"🧾 Total .tif encontrados: {len(tif_files)}")
    print(f"💾 Saída: {out_csv}")

    # Descobrir número de bandas pela primeira imagem
    first = tif_files[0]
    arr0 = read_tif_as_chw(first)       # (C,H,W)
    C0 = arr0.shape[0]

    # Criar header
    header = build_header(C0, patch=patch)

    # Se CSV existe, remove para recriar do zero (pra evitar misturar execuções)
    if out_csv.exists():
        out_csv.unlink()
        print("⚠️ CSV existente removido para recriar:", out_csv.name)

    # Escrever header
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)

    # Processar arquivos
    n_pix = patch * patch
    expected_len = 2 + C0 * n_pix  # image_id, filepath, pixels

    for idx, p in enumerate(tqdm(tif_files, desc=f"Processando {folder.name}", unit="img")):
        try:
            arr = read_tif_as_chw(p)   # (C,H,W)
            if arr.shape[0] != C0:
                # Se mudar o nº de bandas, você pode: pular ou adaptar
                print(f"\n⚠️ Pulando {p.name}: bandas diferentes ({arr.shape[0]} != {C0})")
                continue

            arr = center_crop_or_pad(arr, patch=patch)  # (C,256,256)
            flat = arr.reshape(C0, -1)                  # (C, 65536)

            # Linha: [id, filepath, ...pixels...]
            row = [idx, str(p)]
            # concatena bandas na ordem b0 depois b1...
            row.extend(flat.flatten(order="C").tolist())

            if len(row) != expected_len:
                print(f"\n⚠️ Linha com tamanho inesperado em {p.name}: {len(row)} vs {expected_len}")
                continue

            # Append incremental (não trava e você acompanha o progresso)
            with open(out_csv, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(row)

            # Logs periódicos
            if (idx + 1) % log_every == 0:
                size_mb = out_csv.stat().st_size / (1024**2)
                print(f"\n✅ {idx+1}/{len(tif_files)} salvas | CSV ~ {size_mb:.2f} MB | Último: {p.name}")

        except Exception as e:
            print(f"\n❌ Erro em {p}: {e}")

        # Limpeza de memória
        del arr
        gc.collect()

    print(f"\n🏁 Finalizado: {out_csv} (tamanho: {out_csv.stat().st_size/(1024**2):.2f} MB)")
    return out_csv

## 5) Execucao por Sensor

Dispara o pipeline para LANDSAT, MODIS e SENTINEL.

Dica: use `limit` para teste rapido antes da execucao completa.


In [ ]:
landsat_csv = OUT_DIR / "landsat_256_flat.csv"
folder_to_csv(
    folder=LANDSAT_DIR,
    out_csv=landsat_csv,
    patch=256,
    pattern="*.tif",
    limit=None,     # coloque um número (ex: 100) pra testar rápido
    log_every=10
)

In [ ]:
modis_csv = OUT_DIR / "modis_256_flat.csv"
folder_to_csv(
    folder=MODIS_DIR,
    out_csv=modis_csv,
    patch=256,
    pattern="*.tif",
    limit=None,
    log_every=10
)

In [ ]:
sentinel_csv = OUT_DIR / "sentinel_256_flat.csv"
folder_to_csv(
    folder=SENTINEL_DIR,
    out_csv=sentinel_csv,
    patch=256,
    pattern="*.tif",
    limit=None,
    log_every=10
)

## 6) Inspecao Rapida dos CSVs

Leitura parcial para confirmar:
- existencia do arquivo
- tamanho esperado
- shape de amostra
- formato do cabecalho


In [22]:
import pandas as pd

for p in [sentinel_csv, modis_csv]:
    if p.exists():
        print("\n📌", p.name)
        print("Tamanho (MB):", p.stat().st_size/(1024**2))
        df_head = pd.read_csv(p, nrows=3)
        print("Shape (3 linhas):", df_head.shape)
        display(df_head.iloc[:, :8])  # mostra só as primeiras colunas pra não travar
    else:
        print("\n❌ Não encontrei:", p)


📌 sentinel_256_flat.csv
Tamanho (MB): 6717.213872909546
Shape (3 linhas): (3, 786434)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5
0,0,/content/drive/MyDrive/SpectraAI/data_raw/S2_c...,0.0,0.0,0.0,0.0,0.0,0.0
1,1,/content/drive/MyDrive/SpectraAI/data_raw/S2_c...,0.0,0.0,0.0,0.0,0.0,0.0
2,2,/content/drive/MyDrive/SpectraAI/data_raw/S2_c...,275.0,217.0,217.0,217.0,217.0,217.0



📌 modis_256_flat.csv
Tamanho (MB): 4688.635372161865
Shape (3 linhas): (3, 458754)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5
0,0,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,1325.0,1096.0,1096.0,1224.0,1171.0,1171.0
1,1,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,443.0,460.0,460.0,541.0,541.0,591.0
2,2,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,671.0,664.0,664.0,664.0,604.0,604.0


## 7) Limpeza de Arquivo Corrompido (Opcional)

Bloco util quando uma execucao e interrompida e deixa CSV inconsistente.


In [21]:
from pathlib import Path

bad_file = Path("/content/drive/MyDrive/SpectraAI/datasets_csv/landsat_256_flat.csv")

if bad_file.exists():
    bad_file.unlink()
    print("Arquivo corrompido removido:", bad_file)
else:
    print("Arquivo não encontrado.")

Arquivo corrompido removido: /content/drive/MyDrive/SpectraAI/datasets_csv/landsat_256_flat.csv


## 8) Versao Robusta: Processar Local e Copiar para Drive

A partir daqui, o notebook usa estrategia mais robusta:
1. grava primeiro em disco local (`/content`)
2. copia para o Drive ao final

Motivacao: diminuir erros de I/O e lentidao de escrita direta no Drive.


In [4]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from tqdm import tqdm
import gc
import shutil
import os

## 9) Configuracao da Versao Robusta

Define entrada, saida no Drive e area temporaria local.


In [5]:
LANDSAT_DIR  = Path("/content/drive/MyDrive/SpectraAI/data_raw/LANDSAT_chips")

DRIVE_OUT_DIR = Path("/content/drive/MyDrive/SpectraAI/datasets_csv")
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TMP_DIR = Path("/content/spectra_tmp_csv")
LOCAL_TMP_DIR.mkdir(parents=True, exist_ok=True)

print("LANDSAT_DIR:", LANDSAT_DIR.exists())
print("Saída Drive:", DRIVE_OUT_DIR)
print("Saída local:", LOCAL_TMP_DIR)

LANDSAT_DIR: True
Saída Drive: /content/drive/MyDrive/SpectraAI/datasets_csv
Saída local: /content/spectra_tmp_csv


## 10) Funcoes Utilitarias da Versao Robusta

Reimplementa utilitarios para o fluxo local->Drive, mantendo o mesmo schema de saida.


In [26]:
PATCH = 256

def center_crop_or_pad(img_chw, patch=256):
    C, H, W = img_chw.shape

    pad_h = max(0, patch - H)
    pad_w = max(0, patch - W)

    if pad_h > 0 or pad_w > 0:
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        img_chw = np.pad(
            img_chw,
            ((0, 0), (top, bottom), (left, right)),
            mode="constant",
            constant_values=0
          )

    _, H, W = img_chw.shape
    start_h = (H - patch) // 2
    start_w = (W - patch) // 2

    return img_chw[:, start_h:start_h+patch, start_w:start_w+patch]

def read_tif(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)  # (C,H,W)
    return arr

def make_columns(num_bands, patch=256):
    cols = ["image_id", "filepath"]
    n = patch * patch
    for b in range(num_bands):
        cols.extend([f"b{b}_p{i}" for i in range(n)])
    return cols

## 11) Funcao Principal (Local -> Drive)

`folder_to_csv_local_then_copy(...)` processa em lotes (`batch_size`) e registra progresso.

Boas praticas usadas:
- flush por lote para reduzir uso de memoria
- `gc.collect()` entre lotes
- tratamento de excecao por arquivo


In [27]:
def folder_to_csv_local_then_copy(folder, local_csv, drive_csv, patch=256, batch_size=5, limit=None):
    folder = Path(folder)
    local_csv = Path(local_csv)
    drive_csv = Path(drive_csv)

    tif_files = sorted(folder.rglob("*.tif"))
    if limit is not None:
        tif_files = tif_files[:limit]

    if len(tif_files) == 0:
        raise ValueError(f"Nenhum .tif encontrado em {folder}")

    print(f"\n📂 Pasta: {folder}")
    print(f"🧾 Total de imagens: {len(tif_files)}")

    first_arr = read_tif(tif_files[0])
    num_bands = first_arr.shape[0]
    columns = make_columns(num_bands, patch)

    if local_csv.exists():
        local_csv.unlink()
        print("Arquivo local antigo removido:", local_csv.name)

    # escreve cabeçalho
    pd.DataFrame(columns=columns).to_csv(local_csv, index=False)
    print("✅ Cabeçalho criado localmente:", local_csv)

    batch_rows = []
    saved_count = 0

    for idx, path in enumerate(tqdm(tif_files, desc=f"Processando {folder.name}", unit="img")):
        try:
            arr = read_tif(path)

            if arr.shape[0] != num_bands:
                print(f"\n⚠️ Pulando {path.name}: bandas diferentes ({arr.shape[0]} vs {num_bands})")
                continue

            arr = center_crop_or_pad(arr, patch=patch)
            flat = arr.reshape(-1)

            row = [idx, str(path)] + flat.tolist()
            batch_rows.append(row)

            if len(batch_rows) >= batch_size:
                df_batch = pd.DataFrame(batch_rows, columns=columns)
                df_batch.to_csv(local_csv, mode="a", header=False, index=False)

                saved_count += len(batch_rows)
                size_mb = local_csv.stat().st_size / (1024**2)
                print(f"\n✅ {saved_count}/{len(tif_files)} imagens salvas | CSV local ~ {size_mb:.2f} MB")

                batch_rows = []
                del df_batch
                gc.collect()

        except Exception as e:
            print(f"\n❌ Erro em {path.name}: {e}")

    if batch_rows:
        df_batch = pd.DataFrame(batch_rows, columns=columns)
        df_batch.to_csv(local_csv, mode="a", header=False, index=False)
        saved_count += len(batch_rows)
        print(f"\n✅ Último bloco salvo | total: {saved_count}")
        del df_batch
        gc.collect()

    print("\n🔎 Validando arquivo local...")
    with open(local_csv, "rb") as f:
        first_bytes = f.read(200)

    print("Primeiros bytes:", first_bytes[:200])

    if first_bytes == b"":
        raise RuntimeError("O arquivo local ficou vazio/corrompido. Processo interrompido.")

    print("📦 Copiando para o Drive...")
    shutil.copy2(local_csv, drive_csv)

    print(f"🏁 Finalizado")
    print(f"Local : {local_csv} | {local_csv.stat().st_size / (1024**2):.2f} MB")
    print(f"Drive : {drive_csv} | {drive_csv.stat().st_size / (1024**2):.2f} MB")

## 12) Teste Rapido de Sanidade

Executa em poucas imagens (`limit=2`) para validar o pipeline antes do processamento completo.


In [29]:
folder_to_csv_local_then_copy(
    folder=LANDSAT_DIR,
    local_csv=LOCAL_TMP_DIR / "landsat_teste.csv",
    drive_csv=DRIVE_OUT_DIR / "landsat_teste.csv",
    patch=256,
    batch_size=3,
    limit=2
)


📂 Pasta: /content/drive/MyDrive/SpectraAI/data_raw/LANDSAT_chips
🧾 Total de imagens: 2
Arquivo local antigo removido: landsat_teste.csv
✅ Cabeçalho criado localmente: /content/spectra_tmp_csv/landsat_teste.csv


Processando LANDSAT_chips: 100%|██████████| 2/2 [00:02<00:00,  1.32s/img]



✅ Último bloco salvo | total: 2

🔎 Validando arquivo local...
Primeiros bytes: b'image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,b0_p8,b0_p9,b0_p10,b0_p11,b0_p12,b0_p13,b0_p14,b0_p15,b0_p16,b0_p17,b0_p18,b0_p19,b0_p20,b0_p21,b0_p22,b0_p23,b0_p24,b0_p25,b0_p26,b0_'
📦 Copiando para o Drive...
🏁 Finalizado
Local : /content/spectra_tmp_csv/landsat_teste.csv | 21.38 MB
Drive : /content/drive/MyDrive/SpectraAI/datasets_csv/landsat_teste.csv | 21.38 MB


## 13) Validacao do Arquivo de Teste

Confere existencia, bytes iniciais e leitura parcial via pandas.


In [30]:
test_path = DRIVE_OUT_DIR / "landsat_teste.csv"

print("Existe?", test_path.exists())
print("Tamanho (MB):", test_path.stat().st_size / (1024**2))

with open(test_path, "rb") as f:
    print("Primeiros bytes:", f.read(200))

df_test = pd.read_csv(test_path, nrows=2)
print("Shape:", df_test.shape)
display(df_test.iloc[:, :10])

Existe? True
Tamanho (MB): 21.382250785827637
Primeiros bytes: b'image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,b0_p8,b0_p9,b0_p10,b0_p11,b0_p12,b0_p13,b0_p14,b0_p15,b0_p16,b0_p17,b0_p18,b0_p19,b0_p20,b0_p21,b0_p22,b0_p23,b0_p24,b0_p25,b0_p26,b0_'
Shape: (2, 458754)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7
0,0,/content/drive/MyDrive/SpectraAI/data_raw/LAND...,0.042248,0.056465,0.063395,0.06114,0.057152,0.052615,0.053413,0.064385
1,1,/content/drive/MyDrive/SpectraAI/data_raw/LAND...,0.000000,0.000000,0.042633,0.03650,0.037160,0.037627,0.041340,0.042275


## 14) Execucao Completa (LANDSAT)

Executa o pipeline sem limite para gerar o dataset final do sensor LANDSAT.


In [ ]:
folder_to_csv_local_then_copy(
    folder=LANDSAT_DIR,
    local_csv=LOCAL_TMP_DIR / "landsat_256_flat.csv",
    drive_csv=DRIVE_OUT_DIR / "landsat_256_flat.csv",
    patch=256,
    batch_size=3,
    limit=None
)

## 15) Checagem Final de Integridade

A celula final confere os CSVs finais (LANDSAT, MODIS, SENTINEL):
- existencia
- tamanho
- leitura parcial
- estrutura basica


In [6]:
files = [
    DRIVE_OUT_DIR / "landsat_256_flat.csv",
    DRIVE_OUT_DIR / "modis_256_flat.csv",
    DRIVE_OUT_DIR / "sentinel_256_flat.csv",
]

for p in files:
    print("\n" + "="*90)
    print("Arquivo:", p.name)
    print("Existe?", p.exists())

    if p.exists():
        print("Tamanho (MB):", p.stat().st_size / (1024**2))
        with open(p, "rb") as f:
            print("Primeiros bytes:", f.read(120))

        df = pd.read_csv(p, nrows=2)
        print("Shape parcial:", df.shape)
        display(df.iloc[:, :10])


Arquivo: landsat_256_flat.csv
Existe? True
Tamanho (MB): 12800.346963882446
Primeiros bytes: b'image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,b0_p8,b0_p9,b0_p10,b0_p11,b0_p12,b0_p13,b0_p14,b0_p15,'
Shape parcial: (2, 458754)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7
0,0,/content/drive/MyDrive/SpectraAI/data_raw/LAND...,0.042248,0.056465,0.063395,0.06114,0.057152,0.052615,0.053413,0.064385
1,1,/content/drive/MyDrive/SpectraAI/data_raw/LAND...,0.000000,0.000000,0.042633,0.03650,0.037160,0.037627,0.041340,0.042275



Arquivo: modis_256_flat.csv
Existe? True
Tamanho (MB): 4688.635372161865
Primeiros bytes: b'image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,b0_p8,b0_p9,b0_p10,b0_p11,b0_p12,b0_p13,b0_p14,b0_p15,'
Shape parcial: (2, 458754)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7
0,0,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,1325.0,1096.0,1096.0,1224.0,1171.0,1171.0,1147.0,1147.0
1,1,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,443.0,460.0,460.0,541.0,541.0,591.0,591.0,593.0



Arquivo: sentinel_256_flat.csv
Existe? True
Tamanho (MB): 6717.213872909546
Primeiros bytes: b'image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,b0_p8,b0_p9,b0_p10,b0_p11,b0_p12,b0_p13,b0_p14,b0_p15,'
Shape parcial: (2, 786434)


,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7
0,0,/content/drive/MyDrive/SpectraAI/data_raw/S2_c...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,/content/drive/MyDrive/SpectraAI/data_raw/S2_c...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Importancia para o Modelo

1. Pipeline consistente reduz erro de entrada no treinamento.
2. Patch e schema fixos simplificam montagem de tensores.
3. Integridade do CSV evita falhas silenciosas em treino/avaliacao.

## Limitacoes Conhecidas
1. CSV flatten pode ficar muito grande para volumes altos.
2. Sem compactacao, o custo de armazenamento cresce rapidamente.
3. Processamento em Python puro pode ser lento para bases extensas.

Para escala maior, considerar formato colunar/binario (Parquet/TFRecord).
